# 04 — Dataset Creation

**Purpose:** Merge features and labels into a single modeling-ready dataset. Handle missing data, feature engineering, outlier capping, scaling, and train/test split assignment.

**Inputs:**
- `data/processed/features_per_task.parquet`
- `data/processed/labels.csv`

**Outputs:**
- `data/processed/dataset_final.parquet`
- `data/processed/scaler.pkl`

In [ ]:
import sys
sys.path.insert(0, '..')

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer

from src.config import (
    FEATURES_PER_TASK, LABELS_CSV, DATASET_FINAL, SCALER_PKL,
    PERFORMANCE_LABEL_COL, RANDOM_STATE, TEST_SIZE
)

pd.set_option('display.max_columns', 50)

## 1. Load inputs

In [ ]:
features_long = pd.read_parquet(FEATURES_PER_TASK)
labels        = pd.read_csv(LABELS_CSV)

print(f"Features (long): {features_long.shape}")
print(f"Labels:          {labels.shape}")
print(f"Participants in features: {features_long['participant_id'].nunique()}")
print(f"Participants in labels:   {labels['participant_id'].nunique()}")

## 2. Pivot features from long to wide format

In [ ]:
# Identify feature columns (everything except participant_id and task)
feature_cols = [c for c in features_long.columns if c not in ['participant_id', 'task']]

# Pivot: create columns like "findDice_correct_Total_duration_of_fixations"
features_wide = features_long.pivot_table(
    index='participant_id',
    columns='task',
    values=feature_cols,
    aggfunc='mean'  # each participant should have one row per task; mean handles any duplicates
)

# Flatten MultiIndex columns: (feature, task) → "task_feature"
features_wide.columns = [f"{task}_{feat}" for feat, task in features_wide.columns]
features_wide = features_wide.reset_index()

print(f"Wide features shape: {features_wide.shape}")
features_wide.head()

## 3. Cross-task aggregate features

In [ ]:
# Mean fixation duration across all tasks
fix_dur_cols = [c for c in features_wide.columns if 'correct_Average_duration_of_fixations' in c]
if fix_dur_cols:
    features_wide['mean_fixation_duration_across_tasks'] = features_wide[fix_dur_cols].mean(axis=1)

# Total correct AOI dwell ratio across tasks
dwell_cols = [c for c in features_wide.columns if 'correct_aoi_dwell_ratio' in c]
if dwell_cols:
    features_wide['total_correct_aoi_dwell_ratio'] = features_wide[dwell_cols].mean(axis=1)

# Mean response time (time to first mouse click on correct AOI) across tasks
rt_cols = [c for c in features_wide.columns if 'correct_Time_to_first_mouse_click' in c]
if rt_cols:
    features_wide['mean_time_to_first_click'] = features_wide[rt_cols].mean(axis=1)

# Mean distractor dwell ratio
distract_cols = [c for c in features_wide.columns if 'distractor_dwell_ratio' in c]
if distract_cols:
    features_wide['mean_distractor_dwell_ratio'] = features_wide[distract_cols].mean(axis=1)

print(f"Shape after aggregate features: {features_wide.shape}")

## 4. Merge with labels

In [ ]:
dataset = labels.merge(features_wide, on='participant_id', how='left')

assert len(dataset) == len(labels), "Rows dropped during merge — check participant ID alignment"
print(f"Dataset shape: {dataset.shape}")
print(f"Participants: {dataset['participant_id'].nunique()}")

## 5. Missing value analysis and imputation

In [ ]:
# Separate feature columns from metadata/label columns
meta_cols = ['participant_id', 'total_score', 'accuracy_rate',
             PERFORMANCE_LABEL_COL, 'speed_label', 'mean_response_time_ms'] + \
            [c for c in dataset.columns if c.endswith('_correct') and len(c) < 30]
meta_cols = [c for c in meta_cols if c in dataset.columns]

X_cols = [c for c in dataset.columns if c not in meta_cols]
print(f"Feature columns: {len(X_cols)}")

null_rate = dataset[X_cols].isnull().mean().sort_values(ascending=False)

# Drop columns with ≥ 30% missing
cols_to_drop = null_rate[null_rate >= 0.30].index.tolist()
print(f"\nDropping {len(cols_to_drop)} columns with ≥30% missing:")
print(cols_to_drop[:20])
dataset = dataset.drop(columns=cols_to_drop)
X_cols  = [c for c in X_cols if c not in cols_to_drop]

# Median imputation for remaining columns
imputer = SimpleImputer(strategy='median')
dataset[X_cols] = imputer.fit_transform(dataset[X_cols])

print(f"\nRemaining nulls after imputation: {dataset[X_cols].isnull().sum().sum()}")

## 6. Outlier capping (1.5×IQR)

In [ ]:
def cap_outliers_iqr(df, cols, factor=1.5):
    df = df.copy()
    for col in cols:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        lower = Q1 - factor * IQR
        upper = Q3 + factor * IQR
        df[col] = df[col].clip(lower, upper)
    return df

dataset = cap_outliers_iqr(dataset, X_cols)
print("Outlier capping applied.")

## 7. Train/test split assignment

In [ ]:
N = len(dataset)
print(f"Total participants: {N}")

# Remove participants with missing performance label
dataset_labeled = dataset.dropna(subset=[PERFORMANCE_LABEL_COL]).copy()
print(f"Participants with valid labels: {len(dataset_labeled)}")

if len(dataset_labeled) < 10:
    print("WARNING: Very few participants. All data will be used for cross-validation only — no hold-out test set.")
    dataset_labeled['split'] = 'train'
else:
    train_idx, test_idx = train_test_split(
        dataset_labeled.index,
        test_size=TEST_SIZE,
        stratify=dataset_labeled[PERFORMANCE_LABEL_COL],
        random_state=RANDOM_STATE
    )
    dataset_labeled['split'] = 'train'
    dataset_labeled.loc[test_idx, 'split'] = 'test'

print("\nSplit distribution:")
print(dataset_labeled['split'].value_counts())

## 8. Feature scaling

In [ ]:
train_mask = dataset_labeled['split'] == 'train'

scaler = StandardScaler()
scaler.fit(dataset_labeled.loc[train_mask, X_cols])

# Save unscaled version
dataset_labeled[[c + '_unscaled' for c in X_cols]] = dataset_labeled[X_cols].values

# Apply scaling
dataset_labeled[X_cols] = scaler.transform(dataset_labeled[X_cols])

joblib.dump(scaler, SCALER_PKL)
print(f"Scaler saved: {SCALER_PKL}")

## 9. Summary and save

In [ ]:
print("=== Dataset Summary ===")
print(f"Shape:              {dataset_labeled.shape}")
print(f"Feature columns:    {len(X_cols)}")
print(f"Participants:       {dataset_labeled['participant_id'].nunique()}")
print(f"High performers:    {(dataset_labeled[PERFORMANCE_LABEL_COL] == 1).sum()}")
print(f"Low performers:     {(dataset_labeled[PERFORMANCE_LABEL_COL] == 0).sum()}")
print(f"Train/test:         {dataset_labeled['split'].value_counts().to_dict()}")

dataset_labeled.to_parquet(DATASET_FINAL, index=False)
print(f"\nSaved: {DATASET_FINAL}")